In [16]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import requests
import time

torch.manual_seed(1337)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 4
block_size = 128
n_embd = 128
dropout = 0.2

In [17]:
# Download Tiny Shakespeare dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text

# Build vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)} # Char to int
itos = {i: ch for i, ch in enumerate(chars)} # Int to char

# Encode and decode functions
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Convert to tensor and split (90/10)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# Get batch function
def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,)) # Random start indices
    x = torch.stack([d[i:i+block_size] for i in ix])       # Input sequence
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])   # Target sequence (shifted by 1)
    x, y = x.to(device), y.to(device)                      # Move to GPU
    return x, y

In [18]:
print(vocab_size)                                  # expect: 65
print(decode(encode("hello world")))               # expect: hello world
print(len(train_data), len(val_data))              # expect: 1003854 111540

xb, yb = get_batch('train')
print(xb.shape, yb.shape)                          # expect: torch.Size([4, 8]) torch.Size([4, 8])
print(torch.equal(xb[:, 1:], yb[:, :-1]))          # expect: True

65
hello world
1003854 111540
torch.Size([4, 128]) torch.Size([4, 128])
True


In [19]:
class Head(nn.Module):
    """ single head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)      # (B, T, head_size)
        q = self.query(x)    # (B, T, head_size)

        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5   # (B, T, T)

        # apply causal mask
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        # apply softmax and save detached weights
        wei = F.softmax(wei, dim=-1)
        self.last_attn = wei.detach()

        # apply dropout and perform the weighted aggregation of the values
        wei = self.dropout(wei)
        v = self.value(x)    # (B, T, head_size)
        out = wei @ v        # (B, T, head_size)
        return out

In [20]:
# Set test batch and block size
batch_size = 4
block_size = 8

torch.manual_seed(1337)
model = BigramLanguageModel(vocab_size)
m = model.to(device)

xb, yb = get_batch('train')
logits, loss = m(xb, yb)

print(logits.shape)                                  # expect: torch.Size([4, 8, 65])
print(sum(p.numel() for p in m.parameters()))        # expect: 4225
print(loss)                                          # expect: a number around 4.x

_, loss_no_target = m(xb)
print(loss_no_target)                                # expect: None

torch.Size([32, 65])
4225
tensor(4.7051, grad_fn=<NllLossBackward0>)
None


In [21]:
torch.manual_seed(0)
head = Head(32).to(device)

x = torch.randn(2, block_size, n_embd, device=device)
out = head(x)

print(out.shape)                                              # expect: torch.Size([2, 128, 32])
print(sum(p.numel() for p in head.parameters()))              # expect: 12288
print(torch.triu(head.last_attn[0], diagonal=1).abs().max())  # expect: 0
print(head.last_attn[0].sum(dim=-1))                          # expect: every row adds up to 1

torch.Size([2, 8, 32])
12288
tensor(0.)
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [22]:
torch.manual_seed(0)
head = Head(32).to(device)
x = torch.randn(2, block_size, n_embd, device=device)
out = head(x)

print(out.shape)                                              # expect: torch.Size([2, 128, 32])
print(sum(p.numel() for p in head.parameters()))              # expect: 12288
print(torch.triu(head.last_attn[0], diagonal=1).abs().max())  # expect: 0
print(head.last_attn[0].sum(dim=-1))                          # expect: every row adds up to 1

torch.Size([2, 8, 32])
12288
tensor(0.)
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [23]:
class Head(nn.Module):
    """ single head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)      # (B, T, head_size)
        q = self.query(x)    # (B, T, head_size)

        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5   # (B, T, T)

        # apply causal mask
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        # apply softmax and save detached weights
        wei = F.softmax(wei, dim=-1)
        self.last_attn = wei.detach()

        # apply dropout and perform the weighted aggregation of the values
        wei = self.dropout(wei)
        v = self.value(x)    # (B, T, head_size)
        out = wei @ v        # (B, T, head_size)
        return out

In [24]:
torch.manual_seed(0)
head = Head(32).to(device)

x = torch.randn(2, block_size, n_embd, device=device)
out = head(x)

print(out.shape)
print(sum(p.numel() for p in head.parameters()))
print(torch.triu(head.last_attn[0], diagonal=1).abs().max())
print(head.last_attn[0].sum(dim=-1))

torch.Size([2, 8, 32])
12288
tensor(0.)
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [25]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # concatenate outputs from all heads along the last dimension
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        # apply projection and dropout
        out = self.dropout(self.proj(out))
        return out


class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [26]:
torch.manual_seed(0)
x = torch.randn(2, block_size, n_embd, device=device)

mha = MultiHeadAttention(num_heads=4, head_size=32).to(device)
print(mha(x).shape)
print(sum(p.numel() for p in mha.parameters()))

ffwd = FeedForward(n_embd).to(device)
print(ffwd(x).shape)
print(sum(p.numel() for p in ffwd.parameters()))

torch.Size([2, 8, 128])
65664
torch.Size([2, 8, 128])
131712


In [27]:
class Block(nn.Module):
    """ a transformer block with pre-norm, self-attention, and feed-forward """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # attention with residual connection
        x = x + self.sa(self.ln1(x))
        # feed-forward with residual connection
        x = x + self.ffwd(self.ln2(x))
        return x

In [28]:
torch.manual_seed(0)
x = torch.randn(2, block_size, n_embd, device=device)

block = Block(n_embd, n_head=4).to(device)
print(block(x).shape)
print(sum(p.numel() for p in block.parameters()))

torch.Size([2, 8, 128])
197888


In [29]:
n_head = 4
n_layer = 4

class TransformerLanguageModel(nn.Module):
    """ a small transformer language model """

    def __init__(self):
        super().__init__()
        # token lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # position lookup table
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # stack of transformer blocks
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        # final layer norm
        self.ln_f = nn.LayerNorm(n_embd)
        # output projection to vocabulary size
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # get token and position embeddings, then add them
        tok_emb = self.token_embedding_table(idx)                  # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device)) # (T, n_embd)
        x = tok_emb + pos_emb                                      # (B, T, n_embd)

        # pass through blocks and final layer norm
        x = self.blocks(x)                                         # (B, T, n_embd)
        x = self.ln_f(x)                                           # (B, T, n_embd)

        # project to vocabulary size
        logits = self.lm_head(x)                                   # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits_flat = logits.view(B * T, C)
            targets_flat = targets.view(B * T)
            loss = F.cross_entropy(logits_flat, targets_flat)

        return logits, loss

In [30]:
torch.manual_seed(0)
batch_size = 4
model = TransformerLanguageModel().to(device)
xb, yb = get_batch('train')

logits, loss = model(xb, yb)
print(logits.shape)
print(sum(p.numel() for p in model.parameters()))
print(loss.item())

logits, loss = model(xb[:, :10])
print(logits.shape, loss)

torch.Size([4, 8, 65])
809537
4.4501872062683105
torch.Size([4, 8, 65]) None


In [31]:
class LSTMLanguageModel(nn.Module):
    """ a language model based on LSTM """
    def __init__(self, vocab_size, n_embd, hidden_size=256, num_layers=2, dropout=0.2):
        super().__init__()
        # token lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # LSTM layer
        self.lstm = nn.LSTM(
            input_size=n_embd,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.dropout = nn.Dropout(dropout)
        # output projection to vocabulary size
        self.lm_head = nn.Linear(hidden_size, vocab_size)

    def forward(self, idx, targets=None):
        # get token embeddings
        tok_emb = self.token_embedding_table(idx)                  # (B, T, n_embd)

        # pass through LSTM
        out, _ = self.lstm(tok_emb)                                # (B, T, hidden_size)
        out = self.dropout(out)

        # project to vocabulary size
        logits = self.lm_head(out)                                 # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits_flat = logits.view(B * T, C)
            targets_flat = targets.view(B * T)
            loss = F.cross_entropy(logits_flat, targets_flat)

        return logits, loss

In [32]:
torch.manual_seed(0)
model = LSTMLanguageModel(vocab_size, n_embd).to(device)
xb, yb = get_batch('train')

logits, loss = model(xb, yb)
print(logits.shape)
print(sum(p.numel() for p in model.parameters()))
print(loss.item())

logits, loss = model(xb)
print(loss)

torch.Size([4, 8, 65])
946625
4.169771671295166
None


In [33]:
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()

    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()

    model.train()
    return out


def train_model(model, name, learning_rate):
    model = model.to(device)
    print(f"Training {name}...")
    print(f"Parameters: {sum(p.numel() for p in model.parameters())}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    history = {'step': [], 'train': [], 'val': []}
    train_time = 0.0

    for iter in range(max_iters + 1):
        if iter % eval_interval == 0 or iter == max_iters:
            losses = estimate_loss(model)
            history['step'].append(iter)
            history['train'].append(losses['train'])
            history['val'].append(losses['val'])
            print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

        if iter == max_iters:
            break

        if device == 'cuda':
            torch.cuda.synchronize()
        t0 = time.time()

        xb, yb = get_batch('train')
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if device == 'cuda':
            torch.cuda.synchronize()
        t1 = time.time()
        train_time += (t1 - t0)

    results[name] = {
        'params': sum(p.numel() for p in model.parameters()),
        'train_loss': history['train'][-1],
        'val_loss': history['val'][-1],
        'train_time': train_time,
        'history': history
    }
    print(f"Finished {name}. Total training time: {train_time:.2f}s\n")
    return model

In [34]:
batch_size = 64
max_iters = 2000
eval_interval = 500
eval_iters = 200
results = {}

print(f"block_size={block_size}, batch_size={batch_size}, max_iters={max_iters}, eval_interval={eval_interval}, eval_iters={eval_iters}")

torch.manual_seed(1337)
bigram_model = BigramLanguageModel(vocab_size)
bigram_model = train_model(bigram_model, "Bigram", learning_rate=1e-2)

torch.manual_seed(1337)
lstm_model = LSTMLanguageModel(vocab_size, n_embd)
lstm_model = train_model(lstm_model, "LSTM", learning_rate=1e-3)

torch.manual_seed(1337)
transformer_model = TransformerLanguageModel()
transformer_model = train_model(transformer_model, "Transformer", learning_rate=1e-3)

print("\n--- Bigram Generated Text ---")
print(generate(bigram_model, max_new_tokens=300))

print("\n--- LSTM Generated Text ---")
print(generate(lstm_model, max_new_tokens=300))

print("\n--- Transformer Generated Text ---")
print(generate(transformer_model, max_new_tokens=300))

block_size=8, batch_size=64, max_iters=2000, eval_interval=500, eval_iters=200
Training Bigram...
Parameters: 4225
step 0: train loss 4.7270, val loss 4.7239
step 500: train loss 2.5515, val loss 2.5620
step 1000: train loss 2.4795, val loss 2.5071
step 1500: train loss 2.4715, val loss 2.4916
step 2000: train loss 2.4682, val loss 2.4883
Finished Bigram. Total training time: 7.94s

Training LSTM...
Parameters: 946625
step 0: train loss 4.1692, val loss 4.1690
step 500: train loss 2.0522, val loss 2.1068
step 1000: train loss 1.8994, val loss 1.9927
step 1500: train loss 1.8317, val loss 1.9323
step 2000: train loss 1.7814, val loss 1.9129
Finished LSTM. Total training time: 165.27s

Training Transformer...
Parameters: 809537
step 0: train loss 4.3146, val loss 4.3153
step 500: train loss 2.1079, val loss 2.1502
step 1000: train loss 1.9899, val loss 2.0707
step 1500: train loss 1.9282, val loss 2.0498
step 2000: train loss 1.8941, val loss 2.0278
Finished Transformer. Total training t

In [35]:
@torch.no_grad()
def generate(model, max_new_tokens=500, start_text="\n", seed=42):
    """ generate text from the model """
    # set seed for reproducibility
    torch.manual_seed(seed)

    # switch to evaluation mode
    model.eval()

    # encode start text and move to device
    idx = torch.tensor(encode(start_text), dtype=torch.long, device=device).unsqueeze(0) # (1, T)

    for _ in range(max_new_tokens):
        # crop context to the last block_size tokens
        idx_cond = idx[:, -block_size:]

        # get logits for the last time step
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]  # (1, vocab_size)

        # convert to probabilities
        probs = F.softmax(logits, dim=-1)

        # sample from the distribution
        idx_next = torch.multinomial(probs, num_samples=1)  # (1, 1)

        # append to the sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (1, T+1)

    # switch back to training mode
    model.train()

    # decode and return
    return decode(idx[0].tolist())

In [36]:
torch.manual_seed(0)
tm = TransformerLanguageModel().to(device)

s1 = generate(tm, max_new_tokens=200, start_text="\n", seed=42)
s2 = generate(tm, max_new_tokens=200, start_text="\n", seed=42)
print(len(s1))
print(s1 == s2)
print(repr(s1[:60]))

201
True
"\nv:sDYqLxfJ\nvdxCaNn:wHd&ygW:YG?n.,pSzoKanqyOn'Z.p HVyke?xaaa"


In [37]:
def train_model(model, name, learning_rate):
    """ train the model and save metrics to global results dict """
    model = model.to(device)
    print(f"Training {name}...")
    print(f"Parameters: {sum(p.numel() for p in model.parameters())}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    history = {'step': [], 'train': [], 'val': []}
    train_time = 0.0

    for iter in range(max_iters + 1):
        # evaluate the loss on train and val sets
        if iter % eval_interval == 0 or iter == max_iters:
            print(f"--> Evaluating at step {iter}...")
            losses = estimate_loss(model)
            history['step'].append(iter)
            history['train'].append(losses['train'])
            history['val'].append(losses['val'])
            print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

        if iter == max_iters:
            break

        if device == 'cuda':
            torch.cuda.synchronize()
        t0 = time.time()

        if iter % 10 == 0:
            print(f"iter {iter}")

        xb, yb = get_batch('train')
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if device == 'cuda':
            torch.cuda.synchronize()
        t1 = time.time()
        train_time += (t1 - t0)

    results[name] = {
        'params': sum(p.numel() for p in model.parameters()),
        'train_loss': history['train'][-1],
        'val_loss': history['val'][-1],
        'train_time': train_time,
        'history': history
    }
    print(f"Finished {name}. Total training time: {train_time:.2f}s\n")
    return model

In [38]:
batch_size = 64
eval_interval = 500
eval_iters = 20
results = {}

# 1. Bigram (Fast)
max_iters = 2000
torch.manual_seed(1337)
bigram_model = BigramLanguageModel(vocab_size)
bigram_model = train_model(bigram_model, "Bigram", learning_rate=1e-2)

# 2. LSTM (Slow, reduce iterations)
max_iters = 500
torch.manual_seed(1337)
lstm_model = LSTMLanguageModel(vocab_size, n_embd)
lstm_model = train_model(lstm_model, "LSTM", learning_rate=1e-3)

# 3. Transformer (Fast)
max_iters = 2000
torch.manual_seed(1337)
transformer_model = TransformerLanguageModel()
transformer_model = train_model(transformer_model, "Transformer", learning_rate=1e-3)

print("\n--- Bigram Generated Text ---")
print(generate(bigram_model, max_new_tokens=300))

print("\n--- LSTM Generated Text ---")
print(generate(lstm_model, max_new_tokens=300))

print("\n--- Transformer Generated Text ---")
print(generate(transformer_model, max_new_tokens=300))

Training Bigram...
Parameters: 4225
--> Evaluating at step 0...
step 0: train loss 4.7373, val loss 4.7149
iter 0
iter 10
iter 20
iter 30
iter 40
iter 50
iter 60
iter 70
iter 80
iter 90
iter 100
iter 110
iter 120
iter 130
iter 140
iter 150
iter 160
iter 170
iter 180
iter 190
iter 200
iter 210
iter 220
iter 230
iter 240
iter 250
iter 260
iter 270
iter 280
iter 290
iter 300
iter 310
iter 320
iter 330
iter 340
iter 350
iter 360
iter 370
iter 380
iter 390
iter 400
iter 410
iter 420
iter 430
iter 440
iter 450
iter 460
iter 470
iter 480
iter 490
--> Evaluating at step 500...
step 500: train loss 2.5481, val loss 2.5572
iter 500
iter 510
iter 520
iter 530
iter 540
iter 550
iter 560
iter 570
iter 580
iter 590
iter 600
iter 610
iter 620
iter 630
iter 640
iter 650
iter 660
iter 670
iter 680
iter 690
iter 700
iter 710
iter 720
iter 730
iter 740
iter 750
iter 760
iter 770
iter 780
iter 790
iter 800
iter 810
iter 820
iter 830
iter 840
iter 850
iter 860
iter 870
iter 880
iter 890
iter 900
iter 910
i

### Note on This Run

This short run was just to check whether the components I wrote actually work together: all three models train, the losses go down, and they can all spit out text. It's not a replication of the main notebook. You can tell from the parameter count (809,537 instead of 824,897) that the Transformer was built with a context length of 8 instead of 128, and the LSTM run stopped at step 500, so these numbers aren't really comparable to the main results.

Still, watching a network I built myself push the loss down from above 4 to around 2.0 and actually start producing real-looking words gave me a real sense that it had finally all clicked together. That was way more satisfying than just looking at someone else's finished code.